In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math


from RK4 import * 

from SWESBP_2d import * 

In [ ]:
import timeit

#plt.switch_backend("TkAgg")          # plots in external window
# plt.switch_backend("nbagg") 
from matplotlib import rcParams
# latex
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{bm}'
# basics
rcParams['lines.linewidth'] = 1.2
rcParams['font.family'] = 'Arial'
rcParams['font.size'] = 22
rcParams['axes.linewidth'] = 0.8
# x-ticks
rcParams['xtick.top'] = True
rcParams['xtick.direction'] = 'in'
rcParams['xtick.minor.visible'] = True
rcParams['xtick.major.size'] = 6
rcParams['xtick.minor.size'] = 3
rcParams['xtick.major.width'] = 1.5
rcParams['xtick.minor.width'] = 1.5
rcParams['xtick.major.pad'] = 5
rcParams['xtick.minor.pad'] = 5
# y-ticks
rcParams['ytick.right'] = True
rcParams['ytick.direction'] = 'in'
rcParams['ytick.minor.visible'] = True
rcParams['ytick.major.size'] = 6
rcParams['ytick.minor.size'] = 3
rcParams['ytick.major.width'] = 1.5
rcParams['ytick.minor.width'] = 1.5
rcParams['ytick.major.pad'] = 5
rcParams['ytick.minor.pad'] = 5
# legend
rcParams['legend.fontsize'] = 15 #rcParams['font.size']
rcParams['legend.labelspacing'] = 0.2
rcParams['legend.loc'] = 'upper left'
rcParams['legend.frameon'] = False
# figure
rcParams['figure.figsize'] = (8.0, 5.0)
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 200
rcParams['savefig.bbox'] = 'tight'


In [ ]:
p1 = SWE_SBP()
p1.acoustic_rate

print(24*60*60*10)

In [ ]:
# Initializations
L = 2 * np.pi #4e5        # length of the domain (km)
t = 0.0          # initial time
tend = 4#.# final time

nx = 51 
ny = 51 #641#1281#2561 #3201#1601#3201      # grid points in x                                                                                                                       
dx = L/(nx-1)
dy = L/(ny-1)# grid increment in x
    # velocity (km/s) (can be an array)                                                                                                             
iplot =20      # snapshot frequency
rho = 1#2.6702     # density [g/cm^3]
K =  2.2 #rho*cs**2 

g = 8 #9.81
H = 8

Ubar= 1# -0.3*np.sqrt(g*H)
Vbar = Ubar
#cs = np.sqrt(K/rho)# shear modulus [GPa]
#Zs = rho * cs
  # shear impedance 

order = 4# order of accuracy
# uy =np.zeros((nx,ny))
# vx = np.zeros((nx,ny))

#Initialize the domain

x = np.zeros((nx, ny))
y = np.zeros((nx, ny))
              
p = np.zeros((nx, ny))    

# Initial particle velocity perturbation and discretize the domain
for i in range(0, nx):
    for j in range(0, ny):
        #F[i, j,0] = np.exp(-np.log(2)*(((i-isx)*dx)**2/(sigma) + ((j-isy)*dy)**2/(sigma)))
        x[i, j] = i*dx
        y[i,j] = j*dy



# Time stepping parameters
cfl = 0.1    # CFL number
dt = (cfl/(10+ np.sqrt(g*H)))*dx                  # Time step
nt = int(round(tend/dt))          # number of time steps
n = 0                             # counter
print(nt)
# Boundary condition reflection coefficients 
# r0 = 0                          # r=0:absorbing, r=1:free-surface, r=-1: clamped 
# r1 = 0                            # r=0:absorbing, r=1:free-surface, r=-1: clamped

# # penalty parameters
# tau_11 = 1 #-1  #\tau_{11}  this is for v0
# tau_12 = 1   #\tau_{1N}}     thisis for p0 #switch tau_11 and tau_22 to -1 if we are using energy flux cons.
# tau_21 = 1    #\tau_{21}    this is for vn
# tau_22 = 1     #\tau_{2N} this is for pn

# Initialize: particle velocity (v); and shear stress (s)
u = np.zeros((nx,ny))
v = np.zeros((nx,ny))
p = np.zeros((nx,ny))

U = np.zeros((nx,ny))
V = np.zeros((nx,ny))
P = np.zeros((nx,ny))

U_t = np.zeros((nx,ny))
V_t = np.zeros((nx,ny))
P_t = np.zeros((nx,ny))

U_x =np.zeros((nx,ny))
V_x = np.zeros((nx,ny))
P_x = np.zeros((nx,ny))

                                

# Difference between analyticla and numerical solutions
EV = [0]                                 # initialize errors in V (velocity)
EU = [0]                                 # initialize errors in U (stress)
T = [0]   
ERROR = [0]
ENSTROPHY_ERROR = [0]          

In [ ]:
from dpsbp_operators_periodic import dxd_m_DP_periodic, dxd_p_DP_periodic, dyd_m_DP_periodic, dyd_p_DP_periodic
from sbp_operators_periodic import dxd_m_SBP_periodic, dyd_m_SBP_periodic
t=0   # initial time

import cmasher as cmr
colormap_name = 'cmr.infinity'
colormap = cmr.get_sub_cmap(colormap_name, 0.0, 1.0)

  # forcing function, forcing = 1,  and no forcing function, forcing = 0

# type of initial data: Gaussian or Sinusoidal
#type_0 = 'KH_Peixoto'
type_0 = 'double_Gaussian'
#type_0 = 'Gaussian'
#type_0 = 'Dam_Brea'
#type_0 = 'Sinusoidal'
#plt.contourf(x,y,u)
if type_0 in ('Sinusoidal'):
            forcing = 1.0  # we must use forcing for Sinusoidal initial condition

# L2-norm normalizer

p1.mms(u, v,p, U_t, V_t, P_t,U_x, V_x,P_t,x, y, t+dt, Ubar,Vbar,type_0 ,nx,ny,dx,dy,order)
A =  (np.linalg.norm(v)) 
B =  (np.linalg.norm(p))



# Loop through time and evolve the wave-fields using ADER time-stepping scheme of N+1 order of accuracy
start = timeit.default_timer()
#type_0 = 'Sinusoidal'
# Generate initial conditions
p1.mms(u, v,p, U_t, V_t, P_t,U_x, V_x,P_t,x, y, t+dt, Ubar,Vbar,type_0 ,nx,ny,dx,dy,order)



f = 8 # 2 * 7.292e-5
uy =np.zeros((nx,ny))
vx = np.zeros((nx,ny))
dyd_p_DP_periodic(uy, u,ny,dy,order)
dxd_p_DP_periodic(vx,v, nx,dx,order)
# dyd_m_SBP_periodic(uy, u,ny,dy,order)
# dxd_m_SBP_periodic(vx,v, nx,dx,order)
vort = ((vx - uy +f ) )  


energy_0=  ((p*(u**2+v**2) /2  + g * p**2).sum()) * dx * dy
enstrophy_0 =  (np.sum(vort**2 * p)) * dx * dy 
# vort = np.abs((uy - vx +f ) )
# # #plt.colorbar()


# plt.pcolormesh(x,y,u)
# plt.colorbar()

# v = 0*v +50
# u = 0*u+50
#plt.colorbar()
#no_time_steps = tend/dt
#plot_intervals =   [0.1*no_time_steps, 0.5*no_time_steps, 1*no_time_steps]
plot_interval_factor = 500
for t in np.arange(0.0, (tend+dt),dt):
    n = n+1
    
    #delta = (y[-1,0]-y[0,0])
    #L = 10
#fd_type 

# SBP, DP, DRP (non-periodic) , SBP_periodic, DP_periodic, DRP_periodic (periodic)
    
    # compute numerical solution 
    acoustic_RK4(p1,u,v, p, u, v, p, rho, K, nx,ny, dx,dy, order,x, y, t, dt,
                     type_0,Ubar,Vbar,H, g,flux_type='nonlinear',vorticity = 'true')
    #def acoustic_RK4(self,ru, rv, rp,u, v, p, rho, K, nx,ny, dx,dy, order, x, y, t, dt, type_0, Ubar,Vbar, H, g):
 # Analytical solution
    p1.mms(U, V,P, U_t, V_t, P_t,U_x, V_x,P_t,x, y, (t+dt), Ubar,Vbar,type_0 ,nx,ny,dx,dy,order)

    #ms(self,U,V, P, U_t, V_t, P_t, V_x, P_x,x, y, t, Ubar,Vbar,type_0,nx,ny):
    #def acoustic_RK4(self,ru, rv, rp,u, v, p, rho, K, nx,ny, dx,dy, order, x, y, t, dt, type_0, U ,V,H,g,fd_type):
    # compute error and append to the error array
    EU.append(np.linalg.norm(U-v)/A)
    EV.append(np.linalg.norm(V-p)/B)
    #print(energy_t)
    # energy_dx = 0.5 * np.sum(dx * (p * u**2 + g * p**2), axis=0)  # Trapezoidal rule for dx
    # energy_dy = 0.5 * np.sum(dy * (energy_dx))  # Trapezoidal rule for dy

     
    dyd_p_DP_periodic(uy, u,ny,dy,order)
    dxd_p_DP_periodic(vx,v, nx,dx,order)
    vort = ((vx - uy +f ) ) 
    
    energy =  ((p*(u**2+v**2) /2  + g * p**2).sum()) * dx * dy
    enstrophy =  (np.sum(vort**2 * p)) * dx * dy 
    ERROR.append((energy - energy_0)/energy_0)
    ENSTROPHY_ERROR.append((enstrophy-enstrophy_0)/enstrophy_0)
    #ERROR.append(error)
    T.append(t)
    if n % plot_interval_factor == 0:
      plt.figure()  # Create a new figure for each contour plot
      plt.pcolormesh(y, x, vort/p, cmap=colormap)  # Modify 'p' to the variable you want to plot
      plt.colorbar(label =r'$\omega/h$')
      #plt.title(f"Contour Plot at t = {t}")
      plt.xlabel(r"$x$")
      plt.ylabel(r"$y$")
      print(f"Contour Plot at t = {t}")
      plt.clim(-0.05,1.25)
      #plt.savefig('wo_{}_poten_vor_.pdf'.format(t))        #plt.clabel('stage')
      plt.show() 



vort = ((vx - uy +f ) )  
dyd_p_DP_periodic(uy, u,ny,dy,order)
dxd_p_DP_periodic(vx,v, nx,dx,order)
# dyd_p_DP_periodic(uy, u,ny,dy,order)
# dxd_p_DP_periodic(vx,v, nx,dx,order)


      

print(ERROR)
#plt.colorbar()
# # plt.show()

#plt.pcolormesh(x,y,vort/p,cmap = 'jet')
# #plt.clim(8,9)
# plt.colorbar()
#plt.show()
# plt.pcolormesh(y, x, vort/p, cmap=colormap)  # Modify 'p' to the variable you want to plot
# plt.colorbar(label =r'$\omega/h$')
# #plt.title(f"Contour Plot at t = {t}")
# plt.xlabel(r"$x$")
# plt.ylabel(r"$y$")
# #plt.clim(-0.05,1.3)
# print(f"Contour Plot at t = {t}")
# #plt.savefig('poten_vort{}.pdf'.format(t))        #plt.clabel('stage')
# plt.show() 
# # plt.show()
# plt.pcolormesh(x,y,p+H)
# plt.colorbar()
# plt.show()
plt.ioff()
# Simulation end time
stop = timeit.default_timer()
print('total simulation time = ', stop - start)                   # print the time required for simulation
print('spatial order | of accuracy = ', order)                                  # print the polynomial degree used
print('number of grid points = ', nx)                     # print the degree of freedom
print('maximum relative error in particle velocity = ', max(EU))  # max. relative error in particle velocity
print('maximum relative error in stress = ', max(EV))             # max. relative error in stress

In [ ]:
# energy_array = np.array(ERROR[1:])

# error = (p-P)/P

# plt.plot(error)
# plt.show()

In [ ]:
plt.pcolormesh(x,y,u,cmap = 'jet')
plt.colorbar()
plt.show()

In [ ]:


import numpy as np
import matplotlib.pyplot as plt

import cmasher as cmr
colormap_name = 'cmr.infinity'
colormap = cmr.get_sub_cmap(colormap_name, 0.0, 1.0)
# Define the constants
f = 8
g = 8

# Create a grid of x and y values
x = np.linspace(0, 2*np.pi, 251)
y = np.linspace(0, 2*np.pi, 251)
x, y = np.meshgrid(x, y)

# Calculate the expression
expression = np.exp(-2.5*((x-np.pi)**2 + (y-2.6*np.pi/3)**2)) + np.exp(-2.5*((x-np.pi)**2 + (y-3.5*np.pi/3)**2))

# Create a contour plot
plt.pcolormesh(y, x, expression, cmap=colormap)
#plt.colorbar()
plt.colorbar(label =r'$\omega/h$')
#plt.title(f"Contour Plot at t = {t}")
plt.xlabel(r"$x$")
plt.ylabel(r"$y$")
# Add labels and title
plt.clim(-0.05,1.25)
#plt.title('Contour Plot of the Expression')
# plt.savefig('init_vort_new.pdf')
# Show the plot
plt.show()


In [ ]:
ERROR
#plt.show()

In [ ]:
plt.pcolormesh(x,y,p,cmap = 'viridis')

In [ ]:
dyd_p_DP_periodic(uy, u,ny,dy,order)
dxd_p_DP_periodic(vx,v, nx,dx,order)


vort = ((vx - uy +f ) )      
plt.pcolormesh(x,y,vort)

In [ ]:
# latex
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{bm}'
# basics
rcParams['lines.linewidth'] = 1.2
rcParams['font.family'] = 'Arial'
rcParams['font.size'] = 22
rcParams['axes.linewidth'] = 0.8
# x-ticks
rcParams['xtick.top'] = True
rcParams['xtick.direction'] = 'in'
rcParams['xtick.minor.visible'] = True
rcParams['xtick.major.size'] = 6
rcParams['xtick.minor.size'] = 3
rcParams['xtick.major.width'] = 1.5
rcParams['xtick.minor.width'] = 1.5
rcParams['xtick.major.pad'] = 5
rcParams['xtick.minor.pad'] = 5
# y-ticks
rcParams['ytick.right'] = True
rcParams['ytick.direction'] = 'in'
rcParams['ytick.minor.visible'] = True
rcParams['ytick.major.size'] = 6
rcParams['ytick.minor.size'] = 3
rcParams['ytick.major.width'] = 1.5
rcParams['ytick.minor.width'] = 1.5
rcParams['ytick.major.pad'] = 5
rcParams['ytick.minor.pad'] = 5
# legend
rcParams['legend.fontsize'] = 20 #rcParams['font.size']
rcParams['legend.labelspacing'] = 0.2
rcParams['legend.loc'] = 'upper left'
rcParams['legend.frameon'] = False
# figure
rcParams['figure.figsize'] = (8.0, 5.0)
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 200
rcParams['savefig.bbox'] = 'tight'

t =  np.arange(0.0, (tend+dt),dt)

plt.plot(t,ERROR[1:],label = 'Energy',lw=3,color = 'k',alpha = 0.8)


plt.plot(t,ENSTROPHY_ERROR[1:],label = 'Enstrophy',lw = 3,ls = 'dashed')
plt.xlabel(r'$t$')
plt.ylabel(r'$\varepsilon_{\textrm{rel}}$')

plt.legend(loc = 'best')

#plt.savefig('energy_enstrophy.pdf')
plt.show()

In [ ]:

plt.xlabel(r'$t$')
plt.ylabel(r'$\')
plt.show()

In [ ]:
!pip install cmasher

In [ ]:
cmap = cmr.get_sub_cmap('cmr.lilac', 0.2, 0.8, N=100)

# View colormap
cmr.view_cmap(cmap)

In [ ]:
np.savetxt('Enstropy_error_hyper.txt',ENSTROPHY_ERROR)

In [ ]:
np.savetxt('t_array_hyper.txt', t)
np.savetxt('ERROR_array_hyper.txt', ERROR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
# latex
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{bm}'
# basics
rcParams['lines.linewidth'] = 1.2
rcParams['font.family'] = 'Arial'
rcParams['font.size'] = 18
rcParams['axes.linewidth'] = 0.8
# x-ticks
rcParams['xtick.top'] = True
rcParams['xtick.direction'] = 'in'
rcParams['xtick.minor.visible'] = True
rcParams['xtick.major.size'] = 6
rcParams['xtick.minor.size'] = 3
rcParams['xtick.major.width'] = 1.5
rcParams['xtick.minor.width'] = 1.5
rcParams['xtick.major.pad'] = 5
rcParams['xtick.minor.pad'] = 5
# y-ticks
rcParams['ytick.right'] = True
rcParams['ytick.direction'] = 'in'
rcParams['ytick.minor.visible'] = True
rcParams['ytick.major.size'] = 6
rcParams['ytick.minor.size'] = 3
rcParams['ytick.major.width'] = 1.5
rcParams['ytick.minor.width'] = 1.5
rcParams['ytick.major.pad'] = 5
rcParams['ytick.minor.pad'] = 5
# legend
rcParams['legend.fontsize'] = 20 #rcParams['font.size']
rcParams['legend.labelspacing'] = 0.2
rcParams['legend.loc'] = 'upper left'
rcParams['legend.frameon'] = False
# figure
rcParams['figure.figsize'] = (8.0, 5.0)
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 200
rcParams['savefig.bbox'] = 'tight'

enstropy_error_hyper = np.loadtxt(r'Enstropy_error_hyper.txt')
enstropy_error = np.loadtxt(r'Enstropy_error.txt')

ERROR_energy_hyper = np.loadtxt(r'ERROR_array_hyper.txt')
ERROR_energy = np.loadtxt(r'ERROR_array.txt')

t_array_hyper = np.loadtxt(r't_array_hyper.txt')
t_array = np.loadtxt(r't_array.txt')

plt.plot(t_array,ERROR_energy_hyper[1:],label = r'Energy, $\alpha = 0.5$',lw=3,color = 'k',alpha = 0.8)
#plt.xlabel(r'$t$')

plt.ylabel(r'Relative change in energy')
plt.savefig('energy_hyper.pdf')
plt.show()
plt.plot(t_array,enstropy_error_hyper[1:] ,label = r'Enstrophy, $\alpha = 0.5$',lw = 3,ls = 'dashed')
#plt.xlabel(r'$t$')

plt.ylabel(r'Relative change in enstrophy')
#plt.legend(loc = 'upper right')
plt.savefig('enstrophy_hyper.pdf')
#plt.plot(t,ERROR_energy_hyper[1:])
plt.show()

plt.plot(t_array,ERROR_energy[1:],label = r'Energy, $\alpha = 0$',lw=3,color = 'k',alpha = 0.8)
plt.xlabel(r'$t$')

plt.ylabel(r'Relative change in energy')

plt.show()
plt.plot(t_array,enstropy_error[1:] ,label = r'Enstrophy, $\alpha = 0$',lw = 3,ls = 'dashed')
plt.xlabel(r'$t$')
plt.ylabel(r'Relative change in enstrophy')
plt.savefig('enstrophy_wohyper.pdf')
plt.show()

#plt.legend()
#
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
# latex
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{bm}'
# basics
rcParams['lines.linewidth'] = 1.2
rcParams['font.family'] = 'Arial'
rcParams['font.size'] = 18
rcParams['axes.linewidth'] = 0.8
# x-ticks
rcParams['xtick.top'] = True
rcParams['xtick.direction'] = 'in'
rcParams['xtick.minor.visible'] = True
rcParams['xtick.major.size'] = 6
rcParams['xtick.minor.size'] = 3
rcParams['xtick.major.width'] = 1.5
rcParams['xtick.minor.width'] = 1.5
rcParams['xtick.major.pad'] = 5
rcParams['xtick.minor.pad'] = 5
# y-ticks
rcParams['ytick.right'] = True
rcParams['ytick.direction'] = 'in'
rcParams['ytick.minor.visible'] = True
rcParams['ytick.major.size'] = 6
rcParams['ytick.minor.size'] = 3
rcParams['ytick.major.width'] = 1.5
rcParams['ytick.minor.width'] = 1.5
rcParams['ytick.major.pad'] = 5
rcParams['ytick.minor.pad'] = 5
# legend
rcParams['legend.fontsize'] = 17 #rcParams['font.size']
rcParams['legend.labelspacing'] = 0.2
rcParams['legend.loc'] = 'upper left'
rcParams['legend.frameon'] = False
# figure
rcParams['figure.figsize'] = (9.0, 6.0)
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 200
rcParams['savefig.bbox'] = 'tight'# Create subplots
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 8), sharex=True)

# Plotting subplots
axes[0, 0].plot(t_array, ERROR_energy[1:], label=r'$\alpha = 0$', lw=3, color='k', alpha=0.6)
axes[0, 0].set_ylabel(r'Relative change in energy')
axes[0, 0].legend(loc = 'upper left')

axes[1, 0].plot(t_array, enstropy_error[1:], label=r'$\alpha = 0$', lw=3, ls='dashed',alpha = 0.6)
axes[1, 0].set_ylabel(r'Relative change in enstrophy')
axes[1, 0].set_xlabel(r'$t$')
axes[1, 0].legend(loc = 'best')

axes[0, 1].plot(t_array, ERROR_energy_hyper[1:], label=r'$\alpha = 0.5$', lw=3, color='k', alpha=0.6)

#axes[0, 1].set_ylabel(r'Relative change in energy')
axes[0, 1].legend(loc = 'upper left')

axes[1, 1].plot(t_array, enstropy_error_hyper[1:], label=r'$\alpha = 0.5$', lw=3, ls='dashed',alpha = 0.6)
axes[1, 1].set_xlabel(r'$t$')
#axes[1, 1].set_ylabel(r'Relative change in enstrophy')
axes[1, 1].legend(loc = 'upper left')

# Adjust layout
plt.tight_layout()
fig.savefig('energy_ens_fin.pdf')